# Building AI Agents with Langchain and LLMs

This guide will walk you through the process of building AI agents using Langchain and Large Language Models (LLMs). We will cover the necessary steps to set up your environment, create an agent, and deploy it for use in later stages.

### 0. Prerequisites

- An Azure account with sufficient permissions to create resources.
- Terraform installed on your local machine.
- Azure CLI installed and configured.
- [Optional] Docker installed for building container images.

### 1. Deploy the LLM in Azure Foundry using Terraform

To demo how to use `Langchain` AI Agents, we will need many resources.
* Azure Foundry to host the LLM model and provide OpenAI-compatible API endpoints.
* Azure Functions or Container Apps to host a Docker container image for an MCP server.
* Cosmos DB to store the agent's memory.
* Dynamic sessions for the agent with Azure Container Apps.

All these resources are defined in the Terraform code provided in the `infra` directory. You can deploy the resources by running the following commands from within the `infra` directory.

In [19]:
! terraform -chdir=infra init -upgrade
! terraform -chdir=infra plan -out tfplan
! terraform -chdir=infra apply tfplan

Initializing the backend...
Initializing provider plugins...
- Finding azure/azapi versions matching ">= 2.8.0"...
- Finding hashicorp/azurerm versions matching ">= 4.58.0"...
- Using previously-installed azure/azapi v2.10.0
- Using previously-installed hashicorp/azurerm v4.76.0

Terraform has been successfully initialized!

You may now begin working with Terraform. Try running "terraform plan" to see
any changes that are required for your infrastructure. All Terraform commands
should now work.

If you ever set or change modules or backend configuration for Terraform,
rerun this command to reinitialize your working directory. If you forget, other
commands will detect it and remind you to do so if necessary.
data.azurerm_client_config.current: Reading...
azurerm_resource_group.rg: Refreshing state... [id=/subscriptions/dcef7009-6b94-4382-afdc-17eb160d709a/resourceGroups/rg-ai-agents-langchain-400]
data.azurerm_client_config.current: Read complete after 0s [id=Y2xpZW50Q29uZmlncy9jbGllbnR

The following resources should be created.

![Azure Foundry and related resources](./images/resources.png)

### 3. Get Endpoint and API Key for the LLM

Retrieve the FQDN of the LLM and the API key from Terraform outputs.

In [20]:
foundry_endpoint = ! terraform -chdir=infra output -raw foundry_endpoint
foundry_endpoint = foundry_endpoint.n
print("Foundry Endpoint:", foundry_endpoint)

foundry_api_key = ! terraform -chdir=infra output -raw foundry_api_key
foundry_api_key = foundry_api_key.n
print("Foundry API Key:", f"{foundry_api_key[-10:]}...")  # Print only the last 10 characters for security

llm_model_deployment_name = ! terraform -chdir=infra output -raw llm_model_deployment_name
llm_model_deployment_name = llm_model_deployment_name.n
print("LLM Model Deployment Name (ChatGPT):", llm_model_deployment_name)

Foundry Endpoint: https://foundry-400.cognitiveservices.azure.com/
Foundry API Key: AAACOGYQ3t...
LLM Model Deployment Name (ChatGPT): gpt-5.4


### 3. Install required Python packages

Install the required Python packages for working with `langchain` using pip.

In [ ]:
%pip install langchain langgraph langchain-openai langchain-mcp-adapters langchain-azure-dynamic-sessions

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### 4. Set Up the LLM Model client

Create a `ChatOpenAI` model client from `langchain` using the endpoint and API key obtained from Terraform outputs.

In [22]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    base_url=f"{foundry_endpoint}/openai/v1",
    api_key=foundry_api_key,
    model=llm_model_deployment_name,
    streaming=True,
    max_completion_tokens=512
)

### 5. Test the LLM model

Invoke the model with a test prompt to ensure it's working correctly.

In [23]:
from langchain_core.messages import HumanMessage

response = model.stream([HumanMessage(content="Tell me about yourself.")])

for chunk in response:
    print(chunk.content, end="", flush=True)

I’m ChatGPT, an AI assistant created by OpenAI.

I can help with things like:
- answering questions
- explaining concepts
- writing and editing
- brainstorming ideas
- summarizing information
- helping with coding, math, and research-style tasks

A few useful things to know about me:
- I don’t have personal experiences, feelings, or consciousness.
- I generate responses based on patterns in data and the context you give me.
- I can make mistakes, so it’s good to verify important information.
- My usefulness depends a lot on how clearly you describe what you want.

You can use me casually or give me very specific tasks. For example:
- “Explain photosynthesis simply.”
- “Help me write a professional email.”
- “Debug this Python code.”
- “Give me 10 startup name ideas.”

If you want, I can also tell you more specifically:
- what I’m good at
- what my limitations are
- how to prompt me better
- how I work behind the scenes

### 6. Use the LLM model in a Langchain Agent

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

agent = create_agent(
    model=model,
    tools=[],
    middleware=[],
    checkpointer=None,
)

async for step in agent.astream(
    {"messages": [HumanMessage(content="Tell me about yourself")]},
    stream_mode="values"
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Tell me about yourself
================================== Ai Message ==================================

I’m ChatGPT, an AI assistant created by OpenAI.

I can help with things like:
- answering questions
- explaining concepts
- writing and editing
- brainstorming ideas
- coding help
- summarizing information
- translating or rephrasing text

A few useful things to know about me:
- I don’t have feelings, consciousness, or personal experiences.
- I generate responses based on patterns in data I was trained on.
- I can be very helpful, but I can also make mistakes, so important facts should be verified.
- I don’t “know” things the way a person does; I predict useful text based on your input and available context.

If you want, I can also tell you:
- what I’m good at
- what my limitations are
- how to get better answers from me
- a more casual “about me” version
